# TMDB 5000 - Modelo Random Forest para Predição de Popularidade

Este notebook implementa as técnicas de IA:
1. **Normalização** dos dados numéricos via `StandardScaler`
2. **PCA** para redução de dimensionalidade
3. **Random Forest** para prever a popularidade dos filmes

**Critérios de Aceite:**
- Dados escalonados via StandardScaler
- PCA com identificação dos componentes principais
- Modelo treinado e salvo como `models/random_forest_model.pkl`
- Métricas documentadas: Acurácia, F1-Score, Precisão

## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Bibliotecas importadas com sucesso!')
print(f'Scikit-learn version: {__import__("sklearn").__version__}')

## 2. Carregamento dos Dados

> **Pré-requisito:** Baixe os arquivos do dataset TMDB 5000 do Kaggle e coloque-os na pasta `data/`:
> - `data/tmdb_5000_movies.csv`
> - `data/tmdb_5000_credits.csv` (opcional)

In [ ]:
DATA_PATH = '../data/tmdb_5000_movies.csv'
MODELS_PATH = '../models/random_forest_model.pkl'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {DATA_PATH}\n'
        'Faça o download do dataset TMDB 5000 do Kaggle e salve em data/tmdb_5000_movies.csv'
    )

df = pd.read_csv(DATA_PATH)
print(f'Dataset carregado: {df.shape[0]} filmes, {df.shape[1]} colunas')
df.head()

## 3. Exploração Inicial dos Dados

In [ ]:
print('=== Informações do Dataset ===')
df.info()
print('\n=== Estatísticas Descritivas ===')
df.describe()

In [ ]:
print('=== Valores Nulos por Coluna ===')
nulls = df.isnull().sum()
nulls_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_report = pd.DataFrame({'Nulos': nulls, 'Percentual (%)': nulls_pct})
null_report[null_report['Nulos'] > 0]

## 4. Preparação dos Dados

### 4.1 Seleção das Features Numéricas

In [ ]:
# Features numéricas disponíveis no dataset TMDB 5000
FEATURES = ['budget', 'revenue', 'runtime', 'vote_average', 'vote_count']
TARGET = 'popularity'

# Verificar se as colunas existem
available = [c for c in FEATURES + [TARGET] if c in df.columns]
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]

if missing:
    print(f'Colunas ausentes no dataset: {missing}')
    FEATURES = [c for c in FEATURES if c in df.columns]

print(f'Features selecionadas: {FEATURES}')
print(f'Variável alvo: {TARGET}')

### 4.2 Criação da Variável Alvo (Classificação Binária)

A coluna `popularity` é contínua. Vamos convertê-la em categoria binária:
- `1` = Alta popularidade (acima da mediana)
- `0` = Baixa popularidade (abaixo da mediana)

In [ ]:
# Trabalhar com subset limpo
df_model = df[FEATURES + [TARGET]].copy()

# Remover linhas com valores nulos
df_model.dropna(inplace=True)

# Remover registros com budget ou revenue zerado (dados incompletos)
for col in ['budget', 'revenue']:
    if col in df_model.columns:
        df_model = df_model[df_model[col] > 0]

print(f'Registros após limpeza: {len(df_model)}')

# Criar variável alvo binária: popularidade acima da mediana = 1
mediana_popularidade = df_model[TARGET].median()
df_model['popular'] = (df_model[TARGET] > mediana_popularidade).astype(int)

print(f'\nMediana de popularidade: {mediana_popularidade:.2f}')
print(f'Distribuição da variável alvo:')
print(df_model['popular'].value_counts())
print(f'\nBalanceamento: {df_model["popular"].value_counts(normalize=True).mul(100).round(1).to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição da popularidade original
axes[0].hist(df_model[TARGET], bins=50, edgecolor='black', color='steelblue')
axes[0].axvline(mediana_popularidade, color='red', linestyle='--', label=f'Mediana: {mediana_popularidade:.1f}')
axes[0].set_title('Distribuição de Popularidade')
axes[0].set_xlabel('Popularidade')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Distribuição da variável alvo binária
labels = ['Baixa (0)', 'Alta (1)']
counts = df_model['popular'].value_counts().sort_index()
axes[1].bar(labels, counts.values, color=['#d9534f', '#5cb85c'], edgecolor='black')
axes[1].set_title('Classificação de Popularidade')
axes[1].set_ylabel('Quantidade de Filmes')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../models/distribuicao_popularidade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em models/distribuicao_popularidade.png')

## 5. Normalização / Escalonamento dos Dados

Aplicamos `StandardScaler` para normalizar as features numéricas, garantindo média 0 e desvio padrão 1.

In [ ]:
X = df_model[FEATURES].values
y = df_model['popular'].values

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'\nEstatísticas ANTES da normalização:')
pd.DataFrame(X, columns=FEATURES).describe().round(2)

In [ ]:
# Aplicar StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('StandardScaler aplicado com sucesso!')
print(f'\nEstatísticas APÓS normalização:')
df_scaled = pd.DataFrame(X_scaled, columns=FEATURES)
print(df_scaled.describe().round(4))


In [ ]:
# Visualizar distribuição antes e depois da normalização
fig, axes = plt.subplots(2, len(FEATURES), figsize=(4 * len(FEATURES), 8))

for i, feature in enumerate(FEATURES):
    # Antes
    axes[0, i].hist(X[:, i], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{feature}\n(Antes)')
    axes[0, i].set_ylabel('Frequência' if i == 0 else '')

    # Depois
    axes[1, i].hist(X_scaled[:, i], bins=30, color='#5cb85c', edgecolor='black', alpha=0.7)
    axes[1, i].set_title(f'{feature}\n(Após StandardScaler)')
    axes[1, i].set_ylabel('Frequência' if i == 0 else '')

plt.suptitle('Distribuição das Features: Antes e Após Normalização', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/normalizacao_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em models/normalizacao_features.png')

## 6. PCA - Análise de Componentes Principais

Aplicamos PCA para identificar os componentes que explicam a maior parte da variância dos dados.

In [ ]:
# PCA - Redução de Componentes com todos os componentes para análise de variância explicada
pca_full = PCA()
pca_full.fit(X_scaled)

# Variância explicada acumulada
variancia_explicada = pca_full.explained_variance_ratio_
variancia_acumulada = np.cumsum(variancia_explicada)

print('=== Variância Explicada por Componente ===')
for i, (var, acum) in enumerate(zip(variancia_explicada, variancia_acumulada)):
    print(f'PC{i+1}: {var*100:.2f}%  (Acumulado: {acum*100:.2f}%)')

In [ ]:
# Gráfico Scree Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

componentes = range(1, len(FEATURES) + 1)

# Variância individual
axes[0].bar(componentes, variancia_explicada * 100, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_title('Scree Plot - PCA')
axes[0].set_xticks(componentes)
for i, v in enumerate(variancia_explicada):
    axes[0].text(i + 1, v * 100 + 0.5, f'{v*100:.1f}%', ha='center', fontsize=9)

# Variância acumulada
axes[1].plot(componentes, variancia_acumulada * 100, 'o-', color='#d9534f', linewidth=2, markersize=8)
axes[1].axhline(y=95, color='green', linestyle='--', label='95% de variância')
axes[1].axhline(y=90, color='orange', linestyle='--', label='90% de variância')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada (%)')
axes[1].set_title('Variância Explicada Acumulada - PCA')
axes[1].set_xticks(componentes)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/pca_variancia.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em models/pca_variancia.png')

In [ ]:
# Determinar número ótimo de componentes (>= 95% de variância)
n_components_95 = np.argmax(variancia_acumulada >= 0.95) + 1
n_components_90 = np.argmax(variancia_acumulada >= 0.90) + 1

print(f'Componentes para 90% de variância: {n_components_90}')
print(f'Componentes para 95% de variância: {n_components_95}')

# Usar o mínimo necessário para 95% de variância
N_COMPONENTS = n_components_95
print(f'\nN_COMPONENTS selecionado: {N_COMPONENTS}')

In [ ]:
# Aplicar PCA com n_components selecionado
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Shape após PCA: {X_pca.shape}')
print(f'Redução de dimensionalidade: {X_scaled.shape[1]} → {X_pca.shape[1]} componentes')
print(f'Variância total retida: {pca.explained_variance_ratio_.sum()*100:.2f}%')

# Loadings (contribuição de cada feature por componente)
df_loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURES,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)
print('\n=== Loadings PCA (contribuição de cada feature) ===')
df_loadings.round(4)

In [ ]:
# Heatmap dos loadings
plt.figure(figsize=(8, 4))
sns.heatmap(
    df_loadings,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Loading'}
)
plt.title('PCA - Loadings por Feature e Componente', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em models/pca_loadings.png')

## 7. Divisão Treino / Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Manter proporção das classes
)

print(f'Treinamento: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X_pca)*100:.0f}%)')
print(f'Teste:       {X_test.shape[0]} amostras ({X_test.shape[0]/len(X_pca)*100:.0f}%)')
print(f'\nDistribuição das classes (treino):')
classes, contagens = np.unique(y_train, return_counts=True)
for c, n in zip(classes, contagens):
    print(f'  Classe {c}: {n} ({n/len(y_train)*100:.1f}%)')

## 8. Treinamento do Modelo Random Forest

In [ ]:
# Hiperparâmetros do Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print('Modelo Random Forest treinado com sucesso!')
print(f'Número de árvores: {rf_model.n_estimators}')
print(f'Profundidade máxima: {rf_model.max_depth}')

## 9. Avaliação do Modelo

### 9.1 Métricas no Conjunto de Teste

In [ ]:
y_pred = rf_model.predict(X_test)

acuracia  = accuracy_score(y_test, y_pred)
precisao  = precision_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

print('=' * 45)
print('   MÉTRICAS DE AVALIAÇÃO - RANDOM FOREST')
print('=' * 45)
print(f'  Acurácia  : {acuracia:.4f} ({acuracia*100:.2f}%)')
print(f'  Precisão  : {precisao:.4f} ({precisao*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f} ({f1*100:.2f}%)')
print(f'  Recall    : {recall:.4f} ({recall*100:.2f}%)')
print('=' * 45)
print()
print('--- Relatório Completo ---')
print(classification_report(y_test, y_pred, target_names=['Baixa Pop.', 'Alta Pop.']))

### 9.2 Validação Cruzada (5-Fold)

In [ ]:
cv_scores = cross_val_score(rf_model, X_pca, y, cv=5, scoring='f1', n_jobs=-1)

print('=== Validação Cruzada (5-Fold) - F1-Score ===')
for i, score in enumerate(cv_scores):
    print(f'  Fold {i+1}: {score:.4f}')
print(f'\n  Média : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

### 9.3 Matriz de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Baixa Pop.', 'Alta Pop.']
)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusão', fontweight='bold')

# Importância das features (no espaço PCA)
importancias = rf_model.feature_importances_
labels_pc = [f'PC{i+1}' for i in range(N_COMPONENTS)]
sorted_idx = np.argsort(importancias)[::-1]

axes[1].bar(
    [labels_pc[i] for i in sorted_idx],
    importancias[sorted_idx],
    color='steelblue',
    edgecolor='black'
)
axes[1].set_title('Importância dos Componentes PCA', fontweight='bold')
axes[1].set_ylabel('Importância')
for i, v in enumerate(importancias[sorted_idx]):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../models/avaliacao_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em models/avaliacao_modelo.png')

## 10. Salvamento do Modelo

O modelo é salvo junto com o scaler e o PCA para garantir que novas predições usem o mesmo pipeline.

In [ ]:
os.makedirs('../models', exist_ok=True)

# Salvar o pipeline completo: scaler + pca + modelo
pipeline = {
    'scaler': scaler,
    'pca': pca,
    'model': rf_model,
    'features': FEATURES,
    'n_components': N_COMPONENTS,
    'mediana_popularidade': mediana_popularidade,
    'metricas': {
        'acuracia': acuracia,
        'precisao': precisao,
        'f1_score': f1,
        'recall': recall,
        'cv_f1_media': cv_scores.mean(),
        'cv_f1_std': cv_scores.std()
    }
}

joblib.dump(pipeline, MODELS_PATH)
print(f'Modelo salvo em: {MODELS_PATH}')

# Verificar tamanho do arquivo
tamanho = os.path.getsize(MODELS_PATH)
print(f'Tamanho do arquivo: {tamanho / 1024:.1f} KB')

## 11. Verificação do Modelo Salvo

In [ ]:
# Carregar e verificar o modelo salvo
pipeline_loaded = joblib.load(MODELS_PATH)

# Teste com uma amostra do conjunto de teste
amostra = X_test[:5]
pred_original = rf_model.predict(amostra)
pred_carregado = pipeline_loaded['model'].predict(amostra)

print('=== Verificação do Modelo Salvo ===')
print(f'Predições (modelo original):  {pred_original}')
print(f'Predições (modelo carregado): {pred_carregado}')
print(f'\nModelos idênticos: {np.array_equal(pred_original, pred_carregado)}')
print(f'\nMétricas armazenadas no arquivo:')
for k, v in pipeline_loaded['metricas'].items():
    print(f'  {k}: {v:.4f}')

## 12. Resumo Final

### Pipeline Implementado

| Etapa | Técnica | Resultado |
|-------|---------|----------|
| Normalização | `StandardScaler` | Média=0, DP=1 |
| Redução Dimensional | `PCA` | ≥95% variância retida |
| Classificação | `RandomForestClassifier` | 200 árvores, profundidade=10 |

### Critérios de Aceite Atendidos

- ✅ Dados escalonados via `StandardScaler`
- ✅ PCA executado e componentes principais identificados
- ✅ Modelo treinado e salvo em `models/random_forest_model.pkl`
- ✅ Métricas documentadas: Acurácia, F1-Score, Precisão, Recall
- ✅ Validação cruzada (5-Fold) realizada
- ✅ Pipeline completo (scaler + PCA + modelo) salvo para reprodutibilidade

In [ ]:
print('\n' + '='*50)
print('      RESUMO FINAL DO MODELO')
print('='*50)
print(f'  Dataset: TMDB 5000 Movies')
print(f'  Amostras utilizadas: {len(df_model)}')
print(f'  Features originais: {len(FEATURES)}')
print(f'  Componentes PCA: {N_COMPONENTS} (≥95% variância)')
print(f'  Variância retida: {pca.explained_variance_ratio_.sum()*100:.2f}%')
print()
print(f'  MÉTRICAS (conjunto de teste 20%):')
print(f'    Acurácia : {acuracia*100:.2f}%')
print(f'    Precisão : {precisao*100:.2f}%')
print(f'    F1-Score : {f1*100:.2f}%')
print(f'    Recall   : {recall*100:.2f}%')
print()
print(f'  CV F1-Score: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'  Modelo salvo em: models/random_forest_model.pkl')
print('='*50)